In [ ]:
!pip uninstall -y mediapipe
!pip install -q mediapipe==0.10.30 opencv-python-headless pandas numpy tqdm

In [ ]:
!wget -q -O pose_landmarker_full.task \
    https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task
!ls -lh pose_landmarker_full.task

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/KP-TA MCI/02. Figshare/Videos"
OUT_CSV   = "/content/drive/MyDrive/KP-TA MCI/03. Week 1/keluaran_1.csv"

In [ ]:
from __future__ import annotations

import argparse
from dataclasses import dataclass, field
from pathlib import Path

import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

LEFT_ANKLE = 27
RIGHT_ANKLE = 28
VIS_THRESH = 0.5
BBOX_JUMP_THRESH = 0.25
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".m4v"}

COLUMNS = [
    "Rekaman", "Durasi (detik)", "Frame dengan pose terdeteksi (%)",
    "Frame dengan kedua pergelangan kaki terdeteksi (%)",
    "Confidence rerata", "Catatan",
]


@dataclass
class VideoStats:
    name: str
    fps: float = 30.0
    n_frames: int = 0
    n_pose_detected: int = 0
    n_ankles_detected: int = 0
    vis_sum: float = 0.0
    vis_count: int = 0
    bbox_centers: list = field(default_factory=list)

    @property
    def duration_s(self) -> float:
        return self.n_frames / self.fps if self.fps else 0.0

    @property
    def pct_pose(self) -> float:
        return 100.0 * self.n_pose_detected / self.n_frames if self.n_frames else 0.0

    @property
    def pct_ankles(self) -> float:
        return 100.0 * self.n_ankles_detected / self.n_frames if self.n_frames else 0.0

    @property
    def mean_confidence(self) -> float:
        return self.vis_sum / self.vis_count if self.vis_count else float("nan")

    def note(self) -> str:
        if self.n_frames == 0:
            return "video tidak terbaca"
        notes = []
        if self.pct_pose < 50:
            notes.append("deteksi pose rendah (<50% frame)")
        if self.pct_ankles < 50:
            notes.append("deteksi pergelangan kaki rendah (<50% frame)")
        if len(self.bbox_centers) >= 2:
            centers = np.array(self.bbox_centers)
            jumps = np.linalg.norm(np.diff(centers, axis=0), axis=1)
            if jumps.size and (jumps > BBOX_JUMP_THRESH).sum() / jumps.size > 0.05:
                notes.append("diduga ID switching pasien-pemeriksa (>5% lompatan bbox besar)")
        return "; ".join(notes) if notes else "aman"

    def to_row(self) -> dict:
        conf = self.mean_confidence
        return {
            "Rekaman": self.name,
            "Durasi (detik)": round(self.duration_s, 2),
            "Frame dengan pose terdeteksi (%)": round(self.pct_pose, 2),
            "Frame dengan kedua pergelangan kaki terdeteksi (%)": round(self.pct_ankles, 2),
            "Confidence rerata": round(conf, 4) if not np.isnan(conf) else None,
            "Catatan": self.note(),
        }


def make_landmarker(model_path: str, min_conf: float = 0.5):
    base_options = mp_tasks.BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=min_conf,
        min_pose_presence_confidence=min_conf,
        min_tracking_confidence=min_conf,
    )
    return vision.PoseLandmarker.create_from_options(options)


def process_video(path: Path, model_path: str, min_conf: float = 0.5) -> VideoStats:
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    stats = VideoStats(name=path.stem, fps=fps)

    landmarker = make_landmarker(model_path, min_conf)
    frame_idx = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            stats.n_frames += 1
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            timestamp_ms = int(frame_idx * 1000 / fps)
            frame_idx += 1

            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            if not result.pose_landmarks:
                continue

            stats.n_pose_detected += 1
            lm = result.pose_landmarks[0]  # orang paling menonjol (num_poses=1)

            vis_vals = [p.visibility for p in lm]
            stats.vis_sum += float(np.mean(vis_vals))
            stats.vis_count += 1

            if lm[LEFT_ANKLE].visibility >= VIS_THRESH and lm[RIGHT_ANKLE].visibility >= VIS_THRESH:
                stats.n_ankles_detected += 1

            xs = [p.x for p in lm]
            ys = [p.y for p in lm]
            stats.bbox_centers.append((float(np.mean(xs)), float(np.mean(ys))))
    finally:
        landmarker.close()
        cap.release()

    return stats


def find_videos(root: Path) -> list[Path]:
    return sorted(p for p in root.rglob("*") if p.suffix.lower() in VIDEO_EXTS)


def load_done_names(out_csv: Path) -> set[str]:
    if not out_csv.exists():
        return set()
    df = pd.read_csv(out_csv)
    df = df[df["Rekaman"] != "Median seluruh rekaman"]
    return set(df["Rekaman"].astype(str))


def append_row(out_csv: Path, row: dict) -> None:
    df_row = pd.DataFrame([row], columns=COLUMNS)
    write_header = not out_csv.exists()
    df_row.to_csv(out_csv, mode="a", header=write_header, index=False)


def finalize_with_median(out_csv: Path) -> pd.DataFrame:
    df = pd.read_csv(out_csv)
    df = df[df["Rekaman"] != "Median seluruh rekaman"].copy()
    median_row = {
        "Rekaman": "Median seluruh rekaman",
        "Durasi (detik)": df["Durasi (detik)"].median(),
        "Frame dengan pose terdeteksi (%)": df["Frame dengan pose terdeteksi (%)"].median(),
        "Frame dengan kedua pergelangan kaki terdeteksi (%)": df[
            "Frame dengan kedua pergelangan kaki terdeteksi (%)"
        ].median(),
        "Confidence rerata": df["Confidence rerata"].median(),
        "Catatan": "",
    }
    df_out = pd.concat([df, pd.DataFrame([median_row])], ignore_index=True)
    df_out.to_csv(out_csv, index=False)
    return df_out


def main(argv=None) -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--data-root", type=Path, required=True)
    ap.add_argument("--model-path", type=str, default="pose_landmarker_full.task")
    ap.add_argument("--out-csv", type=Path, default=Path("keluaran1_kualitas_pose.csv"))
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--min-conf", type=float, default=0.5)
    args, _ = ap.parse_known_args(argv)

    videos = find_videos(args.data_root)
    if args.limit:
        videos = videos[: args.limit]
    if not videos:
        raise SystemExit(f"Tidak ada video ditemukan di {args.data_root}")

    done = load_done_names(args.out_csv)
    remaining = [v for v in videos if v.stem not in done]
    print(f"Total video: {len(videos)} | sudah selesai sebelumnya: {len(done)} | akan diproses: {len(remaining)}")

    for vp in tqdm(remaining, desc="Ekstraksi pose"):
        st = process_video(vp, args.model_path, min_conf=args.min_conf)
        append_row(args.out_csv, st.to_row())

    df_out = finalize_with_median(args.out_csv)
    print(f"\nSelesai. Keluaran 1 disimpan di: {args.out_csv.resolve()}")
    print(df_out.to_string(index=False))


if __name__ == "__main__":
    main(__import__("sys").argv[1:])

In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision

CSV_PATH = "/content/drive/MyDrive/KP-TA MCI/03. Week 1/keluaran_1.csv"   # sesuaikan ekstensi
DATA_ROOT = "/content/drive/MyDrive/KP-TA MCI/02. Figshare/Videos"
MODEL_PATH = "pose_landmarker_full.task"
OUT_IMG = "/content/drive/MyDrive/KP-TA MCI/03. Week 1/slide_judul_pose.png"
SAMPLE_STRIDE = 5          # cek tiap 5 frame (percepat pencarian, cukup untuk cari 1 frame terbaik)
VIS_THRESH = 0.5

# 33-landmark BlazePose skeleton connections (untuk menggambar garis rangka)
POSE_CONNECTIONS = [
    (11,12),(11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
    (12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
    (11,23),(12,24),(23,24),
    (23,25),(25,27),(27,29),(27,31),(29,31),
    (24,26),(26,28),(28,30),(28,32),(30,32),
    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),
    (9,10),
]

# 1. Pilih video terbaik berdasarkan skor gabungan
df = pd.read_csv(CSV_PATH)
df = df[df["Rekaman"] != "Median seluruh rekaman"].copy()
df["skor"] = (
    df["Frame dengan pose terdeteksi (%)"]
    + df["Frame dengan kedua pergelangan kaki terdeteksi (%)"]
    + df["Confidence rerata"] * 100
) / 3
df_sorted = df.sort_values("skor", ascending=False)

print("Top 10 kandidat video judul:")
print(df_sorted[["Rekaman", "Frame dengan pose terdeteksi (%)",
                  "Frame dengan kedua pergelangan kaki terdeteksi (%)",
                  "Confidence rerata", "skor"]].head(10).to_string(index=False))

video_terpilih = df_sorted.iloc[0]["Rekaman"]
print(f"\nVideo terpilih: {video_terpilih}")

# 2. Cari file video
def find_video_path(root, stem):
    root = Path(root)
    for ext in [".mp4", ".avi", ".mov", ".mkv", ".m4v"]:
        matches = list(root.rglob(f"{stem}{ext}"))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"Video {stem} tidak ditemukan di {root}")

video_path = find_video_path(DATA_ROOT, video_terpilih)
print(f"Path video: {video_path}")

# 3. Cari frame terbaik di video terpilih
def make_landmarker(model_path, min_conf=0.5):
    base_options = mp_tasks.BaseOptions(model_asset_path=model_path)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_poses=1,
        min_pose_detection_confidence=min_conf,
        min_pose_presence_confidence=min_conf,
        min_tracking_confidence=min_conf,
    )
    return vision.PoseLandmarker.create_from_options(options)

CORE_LANDMARKS = [11, 12, 23, 24, 25, 26, 27, 28]  # bahu, pinggul, lutut, pergelangan kaki
TOLERANSI_TEPI = 0.03  # izinkan sedikit meleset di luar [0,1] (estimasi titik dekat tepi)

def cari_frame_terbaik(video_path, model_path, sample_stride=SAMPLE_STRIDE):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    landmarker = make_landmarker(model_path)

    best_strict = {"skor": -1.0, "frame": None, "landmarks": None}   # badan penuh dalam frame
    best_any = {"skor": -1.0, "frame": None, "landmarks": None}      # fallback: confidence tertinggi saja
    frame_idx = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_idx % sample_stride == 0:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
                timestamp_ms = int(frame_idx * 1000 / fps)
                result = landmarker.detect_for_video(mp_image, timestamp_ms)
                if result.pose_landmarks:
                    lm = result.pose_landmarks[0]
                    vis_vals = [p.visibility for p in lm]
                    skor = float(np.mean(vis_vals))

                    if skor > best_any["skor"]:
                        best_any = {"skor": skor, "frame": frame.copy(), "landmarks": lm}

                    # hanya cek titik INTI tubuh, dengan toleransi tepi
                    inti_dalam_frame = all(
                        -TOLERANSI_TEPI <= lm[i].x <= 1 + TOLERANSI_TEPI
                        and -TOLERANSI_TEPI <= lm[i].y <= 1 + TOLERANSI_TEPI
                        for i in CORE_LANDMARKS
                    )
                    if inti_dalam_frame and skor > best_strict["skor"]:
                        best_strict = {"skor": skor, "frame": frame.copy(), "landmarks": lm}
            frame_idx += 1
    finally:
        landmarker.close()
        cap.release()

    if best_strict["frame"] is not None:
        return best_strict
    print("Peringatan: tidak ada frame dengan seluruh titik inti tubuh dalam batas frame — pakai frame confidence tertinggi saja.")
    return best_any

hasil = cari_frame_terbaik(video_path, MODEL_PATH)
if hasil["frame"] is None:
    raise RuntimeError("Tidak ditemukan pose sama sekali pada video ini — coba video lain dari daftar top-10.")
print(f"Confidence rerata frame terbaik: {hasil['skor']:.3f}")

# 4. Gambar overlay skeleton
def gambar_skeleton(frame, landmarks, vis_thresh=VIS_THRESH):
    h, w = frame.shape[:2]
    out = frame.copy()
    pts = [(int(p.x * w), int(p.y * h)) for p in landmarks]
    vis = [p.visibility for p in landmarks]
    for a, b in POSE_CONNECTIONS:
        if vis[a] >= vis_thresh and vis[b] >= vis_thresh:
            cv2.line(out, pts[a], pts[b], (0, 200, 255), 3, cv2.LINE_AA)
    for i, (x, y) in enumerate(pts):
        if vis[i] >= vis_thresh:
            cv2.circle(out, (x, y), 5, (0, 140, 255), -1, cv2.LINE_AA)
            cv2.circle(out, (x, y), 5, (255, 255, 255), 1, cv2.LINE_AA)
    return out

frame_overlay = gambar_skeleton(hasil["frame"], hasil["landmarks"])

# 5. Crop ke rasio 16:9, terpusat pada orang
def crop_16_9_fokus_orang(frame, landmarks, padding=0.35):
    h, w = frame.shape[:2]
    xs = [p.x * w for p in landmarks]
    ys = [p.y * h for p in landmarks]
    x1, x2 = min(xs), max(xs)
    y1, y2 = min(ys), max(ys)
    bw, bh = x2 - x1, y2 - y1
    x1 -= bw * padding; x2 += bw * padding
    y1 -= bh * padding; y2 += bh * padding
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2

    target_h = max(y2 - y1, (x2 - x1) * 9 / 16)
    target_w = target_h * 16 / 9
    nx1, nx2 = cx - target_w / 2, cx + target_w / 2
    ny1, ny2 = cy - target_h / 2, cy + target_h / 2

    # geser supaya tidak keluar batas frame, baru clip
    if nx1 < 0: nx2 -= nx1; nx1 = 0
    if ny1 < 0: ny2 -= ny1; ny1 = 0
    if nx2 > w: nx1 -= (nx2 - w); nx2 = w
    if ny2 > h: ny1 -= (ny2 - h); ny2 = h
    nx1, ny1 = max(0, int(nx1)), max(0, int(ny1))
    nx2, ny2 = min(w, int(nx2)), min(h, int(ny2))
    return frame[ny1:ny2, nx1:nx2]

frame_final = crop_16_9_fokus_orang(frame_overlay, hasil["landmarks"])

# 6. Simpan & tampilkan
cv2.imwrite(OUT_IMG, frame_final)
print(f"Disimpan: {OUT_IMG}")

from google.colab.patches import cv2_imshow
cv2_imshow(frame_final)

In [ ]:
import cv2
import numpy as np

KANDIDAT = df_sorted["Rekaman"].head(8).tolist()   # 8 kandidat teratas dari ranking sebelumnya
THUMB_W = 480

thumbs = []
labels = []
for nama in KANDIDAT:
    try:
        vp = find_video_path(DATA_ROOT, nama)
        h = cari_frame_terbaik(vp, MODEL_PATH)
        if h["frame"] is None:
            continue
        overlay = gambar_skeleton(h["frame"], h["landmarks"])
        crop = crop_16_9_fokus_orang(overlay, h["landmarks"])
        scale = THUMB_W / crop.shape[1]
        thumb = cv2.resize(crop, (THUMB_W, int(crop.shape[0] * scale)))
        cv2.putText(thumb, nama, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4, cv2.LINE_AA)
        cv2.putText(thumb, nama, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
        thumbs.append(thumb)
        labels.append(nama)
    except Exception as e:
        print(f"Lewati {nama}: {e}")

# susun grid 2 kolom
h_min = min(t.shape[0] for t in thumbs)
thumbs = [t[:h_min] for t in thumbs]
rows = [np.hstack(thumbs[i:i+2]) for i in range(0, len(thumbs), 2)]
w_min = min(r.shape[1] for r in rows)
rows = [r[:, :w_min] for r in rows]
grid = np.vstack(rows)

cv2.imwrite("/content/drive/MyDrive/KP-TA MCI/03. Week 1/kandidat_judul_grid.png", grid)
from google.colab.patches import cv2_imshow
cv2_imshow(grid)
print("Periksa satu per satu: pilih nama rekaman yang skeleton-nya BENAR ada di pasien (bertelanjang kaki / ada strap IMU), bukan pemeriksa.")

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision
from google.colab.patches import cv2_imshow

DATA_ROOT = "/content/drive/MyDrive/KP-TA MCI/02. Figshare/Videos"
MODEL_PATH = "pose_landmarker_full.task"

def spot_check(video_name, n_samples=6):
    video_path = f"{DATA_ROOT}/{video_name}.mp4"
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    sample_idxs = np.linspace(0, total - 1, n_samples, dtype=int)

    options = vision.PoseLandmarkerOptions(
        base_options=mp_tasks.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=vision.RunningMode.VIDEO, num_poses=1,
        min_pose_detection_confidence=0.5,
    )
    landmarker = vision.PoseLandmarker.create_from_options(options)

    print(f"\n=== {video_name} ===")
    for idx in sample_idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok:
            continue
        h, w = frame.shape[:2]
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect_for_video(mp_image, int(idx * 1000 / fps))

        vis = frame.copy()
        if result.pose_landmarks:
            for lm in result.pose_landmarks[0]:
                x, y = int(lm.x * w), int(lm.y * h)
                color = (0, 255, 0) if lm.visibility >= 0.5 else (0, 0, 255)
                cv2.circle(vis, (x, y), 6, color, -1)
            # tandai khusus ankle kiri(27)/kanan(28) biar gampang dilihat
            for i in [27, 28]:
                lm = result.pose_landmarks[0][i]
                x, y = int(lm.x * w), int(lm.y * h)
                cv2.circle(vis, (x, y), 10, (255, 255, 0), 2)
        else:
            cv2.putText(vis, "TIDAK ADA POSE", (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        print(f"t={idx/fps:.1f}s")
        cv2_imshow(vis)

    landmarker.close()
    cap.release()

for name in ["PDFE31_2", "PDFE34_1", "PDFE19_2", "PDFE11_2", "PDFE30_1"]:
    spot_check(name)

In [ ]:
!python keluaran1_pose_quality.py \
    --data-root /content/drive/MyDrive/kp2_figshare/Videos \
    --out-csv /content/drive/MyDrive/kp2_figshare/keluaran1_test.csv \
    --model-path pose_landmarker_full.task \